# 가장 단순한 파서 하나로 모든 형식 읽기

**핵심 아이디어** : docx · hwpx · pptx 는 모두 *ZIP 안의 XML* 이고, 태그 이름(네임스페이스를 뺀 이름)이 거의 같다.

| 형식 | 본문 XML | 문단 | 글자 | 표 · 행 · 칸 |
| --- | --- | --- | --- | --- |
| docx | `word/document.xml` | `w:p` | `w:t` | `w:tbl · w:tr · w:tc` |
| hwpx | `Contents/section0.xml` | `hp:p` | `hp:t` | `hp:tbl · hp:tr · hp:tc` |
| pptx | `ppt/slides/slide1.xml` | `a:p` | `a:t` | `a:tbl · a:tr · a:tc` |

→ lxml 의 `{*}p` (네임스페이스 상관없이 `p`) 로 찾으면 **함수 하나**로 세 형식을 모두 처리할 수 있다.

나머지 : pdf 는 pypdf, html 은 lxml.html, md · txt 는 빈 줄로 나누기, xlsx 는 행(`row`) 단위.

**효율** : 파일마다 ZIP 을 한 번만 열고, 파싱은 C 로 만든 lxml 이 한다. python-docx 같은 무거운 객체 모델을 거치지 않는다.

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "Mon" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "W5" / "Mon"
SAMPLES = SANDBOX / "samples"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

## 파서 전체 코드

In [ ]:
import re
import zipfile
from pathlib import Path

import lxml.html
from lxml import etree
from pypdf import PdfReader

from app.integrations.ports import ParsedBlock, ParsedDoc

# ZIP 형식별 본문 XML 파일 이름 규칙
ZIP_BODY = {
    '.docx': r'word/document\.xml',
    '.hwpx': r'Contents/section\d+\.xml',
    '.pptx': r'ppt/slides/slide\d+\.xml',
}

# 이름 안의 숫자를 숫자로 비교 (slide10 이 slide2 뒤에 오도록)
def _natural(name: str) -> list:
    return [int(s) if s.isdigit() else s for s in re.split(r'(\d+)', name)]

# 표 행 목록 -> 마크다운 표 한 덩어리
def _md_table(rows: list[list[str]]) -> str:
    lines = ['| ' + ' | '.join(r) + ' |' for r in rows]
    lines.insert(1, '|' + '---|' * len(rows[0]))
    return '\n'.join(lines)

# 표(tbl) 안에 있는 요소인지
def _in_table(el) -> bool:
    return any(a.tag.endswith('}tbl') for a in el.iterancestors())


# docx · hwpx · pptx 공통 : 문단은 글 블록, 표는 표 블록 (본문 순서 그대로)
def _parse_zip_xml(path: Path) -> ParsedDoc:
    blocks: list[ParsedBlock] = []
    with zipfile.ZipFile(path) as zf:
        names = sorted((n for n in zf.namelist() if re.fullmatch(ZIP_BODY[path.suffix], n)), key=_natural)
        for no, name in enumerate(names, 1):
            where = f'슬라이드{no}' if path.suffix == '.pptx' else path.stem
            for el in etree.fromstring(zf.read(name)).iter('{*}p', '{*}tbl'):
                if _in_table(el):                       # 표 안의 문단·표는 바깥 표에서 한 번에 처리
                    continue
                if el.tag.endswith('}tbl'):
                    rows = [[''.join(tc.itertext()).strip() for tc in tr.iterchildren('{*}tc')]
                            for tr in el.iterchildren('{*}tr')]
                    if rows:
                        blocks.append(ParsedBlock('표', where, _md_table(rows)))
                else:
                    text = ''.join(t.text or '' for t in el.iter('{*}t') if not _in_table(t)).strip()
                    if text:
                        blocks.append(ParsedBlock('조항', where, text))
    pages = len(names) if path.suffix == '.pptx' else 1
    return ParsedDoc(blocks, pages, sum(b.kind == '표' for b in blocks))


# xlsx : 시트 하나 = 표 하나 (글자는 셀 안(inlineStr) 또는 공유 문자열표(s)에 있다)
def _parse_xlsx(path: Path) -> ParsedDoc:
    blocks: list[ParsedBlock] = []
    with zipfile.ZipFile(path) as zf:
        names = zf.namelist()
        shared = []
        if 'xl/sharedStrings.xml' in names:
            shared = [''.join(si.itertext()) for si in etree.fromstring(zf.read('xl/sharedStrings.xml')).iter('{*}si')]
        sheets = sorted((n for n in names if re.fullmatch(r'xl/worksheets/sheet\d+\.xml', n)), key=_natural)
        for no, name in enumerate(sheets, 1):
            rows = [
                [shared[int(c.findtext('{*}v'))] if c.get('t') == 's' else ''.join(c.itertext()).strip()
                 for c in row.iterchildren('{*}c')]
                for row in etree.fromstring(zf.read(name)).iter('{*}row')
            ]
            if rows:
                blocks.append(ParsedBlock('표', f'시트{no}', _md_table(rows)))
    return ParsedDoc(blocks, len(sheets), len(blocks))


# pdf : 페이지 하나 = 블록 하나 (글자가 없으면 스캔본 -> 블록 0개)
def _parse_pdf(path: Path) -> ParsedDoc:
    reader = PdfReader(path)
    blocks = [ParsedBlock('조항', f'p.{i}', t) for i, page in enumerate(reader.pages, 1)
              if (t := (page.extract_text() or '').strip())]
    return ParsedDoc(blocks, len(reader.pages), 0)


# html : 제목·문단·목록은 글, table 은 표
def _parse_html(path: Path) -> ParsedDoc:
    blocks: list[ParsedBlock] = []
    for el in lxml.html.fromstring(path.read_bytes()).iter('h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'p', 'li', 'table'):
        if el.tag == 'table':
            rows = [[c.text_content().strip() for c in tr.iterchildren('th', 'td')] for tr in el.iter('tr')]
            if rows:
                blocks.append(ParsedBlock('표', el.tag, _md_table(rows)))
        elif not any(a.tag == 'table' for a in el.iterancestors()) and (text := el.text_content().strip()):
            blocks.append(ParsedBlock('조항', el.tag, text))
    return ParsedDoc(blocks, 1, sum(b.kind == '표' for b in blocks))


# md · txt : 빈 줄로 나누기
def _parse_text(path: Path) -> ParsedDoc:
    parts = [s.strip() for s in re.split(r'\n\s*\n', path.read_text(encoding='utf-8')) if s.strip()]
    return ParsedDoc([ParsedBlock('조항', f'문단{i}', s) for i, s in enumerate(parts, 1)], 1, 0)


PARSERS = {
    '.docx': _parse_zip_xml, '.hwpx': _parse_zip_xml, '.pptx': _parse_zip_xml,
    '.xlsx': _parse_xlsx,
    '.pdf': _parse_pdf,
    '.html': _parse_html, '.htm': _parse_html,
    '.md': _parse_text, '.txt': _parse_text,
}

# 입구는 하나 : 확장자를 보고 알맞은 파서로 보낸다
def parse(path: str | Path) -> ParsedDoc:
    path = Path(path)
    parser = PARSERS.get(path.suffix.lower())
    if parser is None:
        raise ValueError(f'지원하지 않는 형식: {path.name}')
    return parser(path)

## 전체 파일 돌려보기

In [ ]:
import time

FILES = sorted(p for p in SAMPLES.rglob('*') if p.is_file() and p.suffix.lower() in PARSERS)

started = time.perf_counter()
results = {p: parse(p) for p in FILES}
elapsed = time.perf_counter() - started

print(f'{len(results)}개 파일 · {elapsed:.2f}초')
print()
for p, d in results.items():
    chars = sum(len(b.text) for b in d.blocks)
    note = '  OCR 필요' if d.page_count and not d.blocks else ''
    print(f'{d.page_count:>3}쪽 · 블록 {len(d.blocks):>3} · 표 {d.table_count} · {chars:>6,}자  {p.name}{note}')

## 같은 문서, 세 형식 결과 비교 & 표 하나 보기

In [ ]:
for p in sorted(SAMPLES.glob('DOC-HR-014_*_v2.0.*')):
    d = results[p]
    print(f'{p.suffix:<6} 블록 {len(d.blocks):>3} · 표 {d.table_count} · {sum(len(b.text) for b in d.blocks):,}자')

print()
hwpx = results[next(SAMPLES.glob('DOC-HR-014_*_v2.0.hwpx'))]
print(next(b for b in hwpx.blocks if b.kind == '표').text)

## 기존 local_parsers (python-docx) 와 비교

- 결과가 같은지, 속도는 얼마나 차이 나는지 docx 파일들로 확인

In [ ]:
from app.rag.local_parsers import parse_docx

docx_files = sorted(SAMPLES.glob('*.docx'))

t0 = time.perf_counter()
old = [parse_docx(p) for p in docx_files]
t1 = time.perf_counter()
new = [parse(p) for p in docx_files]
t2 = time.perf_counter()

same = all([b.text for b in o.blocks] == [b.text for b in n.blocks] for o, n in zip(old, new))
print(f'블록 내용 동일 : {same}')
print(f'python-docx : {t1 - t0:.3f}초')
print(f'단순 파서    : {t2 - t1:.3f}초')